# Notebook 01 — Exploratory Data Analysis & Feature Engineering
## FixtureIQ · Multi-Season Data Integration (2022/23 – 2024/25)

### What this notebook does
1. **Loads and unifies** all data sources across 3 seasons:
   - FBref match logs (`*_matches_all.csv`) — team-level match calendar
   - FBref player stats (`*_players_all_competitions.csv`) — season aggregates
   - FBref (`\match_reports/*.csv`) — per match info
   - SofaScore position profiles (`position_profiles/*.csv`) — rich per-position metrics
   - SofaScore player profiles (`premier_league_2022_2023_all_players.csv`) — 
   - Injuries data (`injuries/` folder) 
2. **Descriptive statistics** — mean, std, min, max, CV per variable
3. **Missing value analysis** — visual missingness maps
4. **Outlier detection** — IQR method, flagged and visualised
5. **Distribution analysis** — histograms for all key variables
6. **Correlation matrix & heatmap** — Pearson + Spearman
7. **Radar charts** — player/team performance profiles by position
8. **Lorenz curve & Gini coefficient** — workload inequality between players
9. **New computed attributes** — fatigue indicators, workload stress index, performance composite
10. **Written conclusions** — interpretations after each main section

### File structure expected
```
Data/
├── 2022-2023/
│   ├── fbref/
│   │   ├── arsenal_2022_2023/         → *_matches_all.csv, *_players_all_competitions.csv
│   │   ├── manchester_city_2022_2023/
│   │   ├── liverpool_2022_2023/         (or chelsea / tottenham depending on UCL year)
│   │   └── .../
│   ├── sofascore/
│   │   ├── ...
│   │   └── premier_league/             → *_all_players.csv, position_profiles/
│   │       ├──position_profiles/*.csv
│   │       └── premier_league_2022_2023_all_players.csv
│   └── injuries/                       
├── 2023-2024/   (same structure)
└── 2024-2025/   (same structure)
```

## 0. Setup

In [1]:
# Uncomment first time
# !pip install pandas numpy matplotlib seaborn scipy --quiet

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from scipy import stats
import math

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (14, 5)})
sns.set_palette('husl')

# ── Paths ─────────────────────────────────────────────────────────────────────
# !! ADJUST THIS to your actual data root !!
path = r'C:\Users\ItxaroAizpuruaArcona\Desktop\S_A\Fixture-IQ-playground\Data'
DATA_ROOT = Path(path)
OUTPUT_DIR = DATA_ROOT / 'EDA_OUTPUTS'
FIGURES_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

# ── Season / team config ──────────────────────────────────────────────────────
SEASONS = {
    '2022-2023': '2022_2023',
    '2023-2024': '2023_2024',
    '2024-2025': '2024_2025',
}

# UCL-participating PL teams per season
# Adjust if your data has different teams for 22/23 or 23/24
UCL_TEAMS_BY_SEASON = {
    '2022-2023': ['Chelsea', 'Liverpool', 'Manchester City', 'Tottenham'],
    '2023-2024': ['Arsenal', 'Manchester City', 'Manchester United', 'Newcastle United'],
    '2024-2025': ['Arsenal', 'Liverpool', 'Manchester City', 'Aston Villa'],
}

TEAM_SLUGS = {
    'Arsenal':           'arsenal',
    'Manchester City':   'manchester_city',
    'Liverpool':         'liverpool',
    'Aston Villa':       'aston_villa',
    'Chelsea':           'chelsea',
    'Manchester United': 'manchester_united',
    'Newcastle United':  'newcastle_united',
    'Tottenham':         'tottenham_hotspur',
}

TEAM_COLORS = {
    'Arsenal':           '#EF0107',
    'Liverpool':         '#C8102E',
    'Manchester City':   '#6CABDD',
    'Aston Villa':       '#670E36',
    'Chelsea':           '#034694',
    'Manchester United': '#DA291C',
    'Newcastle United':  '#241F20',
    'Tottenham':         '#132257',
}

print('Setup complete.')
print(f'Data root : {DATA_ROOT}')
print(f'Output    : {OUTPUT_DIR}')

Setup complete.
Data root : C:\Users\ItxaroAizpuruaArcona\Desktop\S_A\Fixture-IQ-playground\Data
Output    : C:\Users\ItxaroAizpuruaArcona\Desktop\S_A\Fixture-IQ-playground\Data\EDA_OUTPUTS


---
## 1. Load Data — FBref Match Logs

In [24]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore') # Hides ffill/copy warnings during loops

# 1. Base Setup
BASE = Path('..')
seasons = ['2022-2023', '2023-2024', '2024-2025']
merge_keys = ['season', 'Match_ID', 'Date', 'Competition', 'Round', 'Venue', 'Opponent', 'Result', 'Team', 'Player']

# High-level lists to hold data across all teams and seasons
all_outfield_rows = []
all_goalkeeper_rows = []

# --- CLEANING HELPER FUNCTIONS ---
def extract_whole_years(age_val):
    if pd.isna(age_val): return np.nan
    if isinstance(age_val, str) and '-' in age_val: return int(age_val.split('-')[0])
    return int(float(age_val))

def clean_common_fields(df, age_lookup, nation_lookup, fallback_median):
    """Standardizes metrics and handles identity lookup fills for benched entries."""
    df['Min'] = df['Min'].fillna(0).astype(int)
    df['played'] = df['Min'] > 0
    df['Age'] = df['Age'].apply(extract_whole_years)
    df['Age'] = df['Age'].fillna(df['Player'].map(age_lookup))
    df['Nation'] = df['Nation'].fillna(df['Player'].map(nation_lookup))
    df['Age'] = df['Age'].fillna(fallback_median).astype(int)
    df['Nation'] = df['Nation'].fillna('Unknown')
    return df


# ==========================================================
# MAIN LOOP: ITERATE THROUGH SEASONS & DETECTED TEAMS
# ==========================================================
for season_str in seasons:
    fbref_path = BASE / 'Data' / season_str / 'fbref'
    
    if not fbref_path.exists():
        print(f"[SKIP] Season folder not found: {season_str}")
        continue
        
    print(f"Processing Season: {season_str}...")
    
    # Scan all child items inside the fbref directory to dynamically detect teams
    for team_dir in sorted(fbref_path.iterdir()):
        if not team_dir.is_dir():
            continue  # Skips stray files like .DS_Store or system logs
            
        # Match report folder mapping
        match_reports_dir = team_dir / 'match_reports'
        
        # Core File Checks
        lineups_path = match_reports_dir / 'master_lineups.csv'
        stats_path = match_reports_dir / 'master_player_stats.csv'
        gk_path = match_reports_dir / 'master_goalkeeper_stats.csv'
        
        # Try to dynamically locate the seasonal profiles registry file
        profile_glob = list(team_dir.glob('*_players_all_competitions.csv'))
        
        # Ensure the essential matchday files exist before running logic
        if not (lineups_path.exists() and stats_path.exists() and gk_path.exists()):
            continue
            
        # --- A. BUILD DICTIONARY LOOKUPS PER TEAM ---
        if profile_glob and profile_glob[0].exists():
            df_seasonal_profile = pd.read_csv(profile_glob[0])
            age_lookup = df_seasonal_profile.set_index('Player')['Age'].to_dict()
            nation_lookup = df_seasonal_profile.set_index('Player')['Nation'].to_dict()
            fallback_median = df_seasonal_profile['Age'].median()
            # Reference map to extract historical positions for benched players later
            pos_lookup = df_seasonal_profile.set_index('Player')['Pos'].to_dict()
        else:
            age_lookup, nation_lookup, pos_lookup = {}, {}, {}
            fallback_median = 26 # Safe general fallback if registry is absent
            
        # --- B. PROCESS MASTER LINEUPS FOUNDATION ---
        df_lineups = pd.read_csv(lineups_path)
        df_master = df_lineups[merge_keys + ['Lineup_Section', 'Formation']].copy()
        df_master['is_in_squad'] = True
        df_master['started'] = df_master['Lineup_Section'] == 'Starter'
        df_master = df_master.drop(columns=['Lineup_Section'])
        
        # --- C. MERGE & PROCESS OUTFIELD PLAYERS ---
        df_stats = pd.read_csv(stats_path)
        outfield_metrics = ['shirtnumber', 'Nation', 'Pos', 'Age', 'Min', 'Gls', 'Ast', 'shots', 'shots_on_target', 'CrdY', 'CrdR', 'fouls', 'fouled', 'offsides', 'crosses', 'tackles_won', 'interceptions']
        df_stats_clean = df_stats[merge_keys + outfield_metrics]
        
        df_outfield_combined = pd.merge(df_master, df_stats_clean, on=merge_keys, how='left')
        df_outfield_combined = clean_common_fields(df_outfield_combined, age_lookup, nation_lookup, fallback_median)
        
        # Clean defaults for outfield columns
        df_outfield_combined['Pos'] = df_outfield_combined['Pos'].fillna('Bench')
        df_outfield_combined['shirtnumber'] = df_outfield_combined['shirtnumber'].fillna(-1).astype(int)
        outfield_numeric = ['Gls', 'Ast', 'shots', 'shots_on_target', 'CrdY', 'CrdR', 'fouls', 'fouled', 'offsides', 'crosses', 'tackles_won', 'interceptions']
        df_outfield_combined[outfield_numeric] = df_outfield_combined[outfield_numeric].fillna(0).astype(int)
        
        # Isolate and save Outfield rows (Drops active GKs)
        df_outfield_team_final = df_outfield_combined[df_outfield_combined['Pos'] != 'GK'].copy()
        all_outfield_rows.append(df_outfield_team_final)
        
        # --- D. MERGE & PROCESS GOALKEEPERS ---
        df_gk = pd.read_csv(gk_path)
        df_gk_clean = df_gk[merge_keys + ['Nation', 'Age', 'Min', 'SoTA', 'GA', 'Saves', 'Save%']]
        
        df_gk_combined = pd.merge(df_master, df_gk_clean, on=merge_keys, how='left')
        df_gk_combined = clean_common_fields(df_gk_combined, age_lookup, nation_lookup, fallback_median)
        
        # Clean defaults for goalkeeper columns
        gk_numeric = ['SoTA', 'GA', 'Saves']
        df_gk_combined[gk_numeric] = df_gk_combined[gk_numeric].fillna(0).astype(int)
        df_gk_combined['Save%'] = df_gk_combined['Save%'].fillna(0.0).astype(float)
        
        # Isolate active GKs and benched GKs using registry cross-check
        gk_names = df_gk['Player'].unique()
        df_gk_team_final = df_gk_combined[
            (df_gk_combined['Player'].isin(gk_names)) | 
            (df_gk_combined['Player'].map(pos_lookup) == 'GK')
        ].copy()
        df_gk_team_final['Pos'] = 'GK'
        
        all_goalkeeper_rows.append(df_gk_team_final)

print("\n--- Compiling Full Aggregates ---")
# Concatenate lists of dataframes into our two master analytical datasets
df_outfield_final = pd.concat(all_outfield_rows, ignore_index=True) if all_outfield_rows else pd.DataFrame()
df_goalkeepers_final = pd.concat(all_goalkeeper_rows, ignore_index=True) if all_goalkeeper_rows else pd.DataFrame()

# ==========================================================
# FINAL VALIDATION REPORT
# ==========================================================
print(f"Total Rows Compiled in df_outfield_final   : {df_outfield_final.shape[0]}")
print(f"Total Rows Compiled in df_goalkeepers_final : {df_goalkeepers_final.shape[0]}")
print("\nUnique Seasons Found:", df_outfield_final['season'].unique())
print("Unique Teams Found:", df_outfield_final['Team'].unique())

Processing Season: 2022-2023...
Processing Season: 2023-2024...
Processing Season: 2024-2025...

--- Compiling Full Aggregates ---
Total Rows Compiled in df_outfield_final   : 11993
Total Rows Compiled in df_goalkeepers_final : 1421

Unique Seasons Found: <ArrowStringArray>
['2022-2023', '2023-2024', '2024-2025']
Length: 3, dtype: str
Unique Teams Found: <ArrowStringArray>
[          'Chelsea',         'Liverpool',   'Manchester City',
 'Tottenham Hotspur',           'Arsenal',    'Manchester Utd',
  'Newcastle United',       'Aston Villa']
Length: 8, dtype: str


In [28]:
print("\nOutfield Preview:")
df_outfield_final.head()


Outfield Preview:


,season,Match_ID,Date,Competition,Round,Venue,Opponent,Result,Team,Player,Formation,is_in_squad,started,shirtnumber,Nation,Pos,Age,Min,Gls,Ast,shots,shots_on_target,CrdY,CrdR,fouls,fouled,offsides,crosses,tackles_won,interceptions,played
0,2022-2023,3a917cee,2022-08-06,Premier League,Matchweek 1,Away,Everton,W,Chelsea,Jorginho,3-4-3,True,True,5,it ITA,CM,30,89,1,0,1,1,0,0,0,1,0,0,0,0,True
1,2022-2023,3a917cee,2022-08-06,Premier League,Matchweek 1,Away,Everton,W,Chelsea,Thiago Silva,3-4-3,True,True,6,br BRA,CB,37,90,0,0,0,0,0,0,0,0,0,0,0,3,True
2,2022-2023,3a917cee,2022-08-06,Premier League,Matchweek 1,Away,Everton,W,Chelsea,N'Golo Kanté,3-4-3,True,True,7,fr FRA,CM,31,90,0,0,1,1,0,0,0,0,1,1,2,2,True
3,2022-2023,3a917cee,2022-08-06,Premier League,Matchweek 1,Away,Everton,W,Chelsea,Raheem Sterling,3-4-3,True,True,17,eng ENG,FW,27,90,0,0,3,0,0,0,3,3,1,0,1,0,True
4,2022-2023,3a917cee,2022-08-06,Premier League,Matchweek 1,Away,Everton,W,Chelsea,Mason Mount,3-4-3,True,True,19,eng ENG,FW,23,64,0,0,2,1,0,0,0,0,0,2,0,0,True


In [27]:
print("\nGoalkeeper Preview:")
df_goalkeepers_final.head()


Goalkeeper Preview:


,season,Match_ID,Date,Competition,Round,Venue,Opponent,Result,Team,Player,Formation,is_in_squad,started,Nation,Age,Min,SoTA,GA,Saves,Save%,played,Pos
0,2022-2023,3a917cee,2022-08-06,Premier League,Matchweek 1,Away,Everton,W,Chelsea,Edouard Mendy,3-4-3,True,True,sn SEN,30,90,3,0,3,100.000,True,GK
1,2022-2023,3a917cee,2022-08-06,Premier League,Matchweek 1,Away,Everton,W,Chelsea,Kepa Arrizabalaga,3-4-3,True,False,esESP,27,0,0,0,0,0.000,False,GK
2,2022-2023,01e57bf5,2022-08-14,Premier League,Matchweek 2,Home,Tottenham Hotspur,D,Chelsea,Edouard Mendy,3-4-3,True,True,sn SEN,30,90,5,2,3,60.000,True,GK
3,2022-2023,01e57bf5,2022-08-14,Premier League,Matchweek 2,Home,Tottenham Hotspur,D,Chelsea,Kepa Arrizabalaga,3-4-3,True,False,esESP,27,0,0,0,0,0.000,False,GK
4,2022-2023,a5632124,2022-08-21,Premier League,Matchweek 3,Away,Leeds United,L,Chelsea,Edouard Mendy,3-5-2,True,True,sn SEN,30,90,6,3,3,50.000,True,GK


---
### FEATURE EXPLANATIONS & ML REPRESENTATIONS:

**1. days_since_last_minutes (Rest Tracker)**
    - Represents: The exact number of days a player has had to physically recover 
        since they last stepped on the pitch for active match minutes.
    - ML Value: Essential for capturing muscular and neural recovery decay. High 
        values imply full recovery or match-rust; low values imply residual fatigue.

**2. acute_workload_7d (Recent Fatigue Proxy)**
    - Represents: Total competitive minutes played over the previous 7 days 
        (excluding the current matchday to prevent data leakage).
    - ML Value: Captures short-term cardiovascular and tissue stress spikes.

**3. chronic_workload_28d (Conditioning / Fitness Baseline)**
    - Represents: The average weekly minutes played over the past 28 days.
    - ML Value: Establishes a baseline of what the player's body is currently 
        conditioned to handle safely without tearing or straining.

**4. acwr (Acute-to-Chronic Workload Ratio)**
    - Represents: Fatigue divided by Fitness (acute_workload_7d / chronic_workload_28d).
    - ML Value: The gold standard metric for injury prediction. 
        - Values between 0.8 and 1.3 = "Sweet Spot" (Safe, high fitness, low risk).
        - Values > 1.5 = "Danger Zone" (Workload spike, high tissue failure risk).

**5. match_count_14d (Fixture Congestion Context)**
    - Represents: Total matches played by the TEAM over the last 14 days.
    - ML Value: Controls for macro-level squad exhaustion, limited training/recovery 
        windows, and general mental/neural burnout from tight schedules.


---


In [44]:
# ==========================================================
# STEP 7: COMPUTE ROLLING FATIGUE AND WORKLOAD METRICS (ML-READY)
# ==========================================================

def engineer_fatigue_features(df_input):
    """
    Computes rolling workloads, acute/chronic ratios, fixture congestion,
    and consecutive rest days per player over time.
    """
    df = df_input.copy()
    
    # Ensure chronological baseline per player for accurate time windows
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(by=['Player', 'Date']).reset_index(drop=True)
    
    # [Metric 1] Rest Tracker: Days since last squad presence vs. actual on-pitch minutes
    df['days_since_last_squad_appearance'] = df.groupby('Player')['Date'].diff().dt.days
    df['last_played_date'] = df['Date'].where(df['played'])
    df['last_played_date'] = df.groupby('Player')['last_played_date'].ffill()
    df['days_since_last_minutes'] = (df['Date'] - df.groupby('Player')['last_played_date'].shift(1)).dt.days
    
    # Fallback to 14 days (fully rested) for initial rows
    df['days_since_last_minutes'] = df['days_since_last_minutes'].fillna(14).astype(int)
    df['days_since_last_squad_appearance'] = df['days_since_last_squad_appearance'].fillna(14).astype(int)
    
    # Set date index to run rolling time windows safely
    df = df.set_index('Date')
    
    # [Metric 2] Acute Workload: Short-term fatigue lookback (Past 7 Days sum)
    df['acute_workload_7d'] = df.groupby('Player')['Min'].rolling('7D', closed='left').sum().reset_index(0, drop=True)
    
    # [Metric 3] Chronic Workload: Long-term conditioning baseline (Past 28 Days weekly average)
    df['chronic_workload_28d'] = (df.groupby('Player')['Min'].rolling('28D', closed='left').sum().reset_index(0, drop=True)) / 4.0
    
    df['acute_workload_7d'] = df['acute_workload_7d'].fillna(0)
    df['chronic_workload_28d'] = df['chronic_workload_28d'].fillna(0)
    
    # [Metric 4] ACWR: Workload spike risk index (Fatigue / Fitness Ratio)
    df['acwr'] = np.where(
        df['chronic_workload_28d'] > 0, 
        df['acute_workload_7d'] / df['chronic_workload_28d'], 
        0.0
    )
    
    df = df.reset_index()
    
    # [Metric 5] Fixture Congestion: General team-level accumulation strain (Past 14 Days match count)
    team_match_dates = df[['Team', 'Date', 'Match_ID']].drop_duplicates().copy()
    team_match_dates = team_match_dates.sort_values(by=['Team', 'Date']).set_index('Date')
    
    team_match_dates['match_count_14d'] = (
        team_match_dates.groupby('Team')['Match_ID']
        .rolling('14D', closed='left').count().reset_index(0, drop=True)
    )
    team_match_dates['match_count_14d'] = team_match_dates['match_count_14d'].fillna(0).astype(int)
    
    df = pd.merge(df, team_match_dates.reset_index()[['Team', 'Match_ID', 'match_count_14d']], on=['Team', 'Match_ID'], how='left')
    
    # Re-sort to chronological matchday diary sequence
    df = df.sort_values(by=['Date', 'Team']).reset_index(drop=True)
    
    return df

# --- APPLY FEATURE ENGINEERING ENGINE TO BOTH DATASETS ---
print("Engineering fatigue markers...")
df_outfield_features = engineer_fatigue_features(df_outfield_final)
df_gk_features = engineer_fatigue_features(df_goalkeepers_final)

# ==========================================================
# CHRONOLOGICAL TEAM-BASED VALIDATION TRACKER
# ==========================================================
# Find the midpoint index of the dataframe
mid_point = len(df_outfield_features) // 2

# Slice 25 rows from the middle
print("=== MIDDLE TIMELINE EXAMPLES ===")
df_outfield_features.iloc[mid_point : mid_point + 25][
    ['Date', 'Team', 'Player', 'Opponent', 'Min', 'days_since_last_minutes', 'acute_workload_7d', 'match_count_14d', 'chronic_workload_28d', 'acwr' ]
]

Engineering fatigue markers...
=== MIDDLE TIMELINE EXAMPLES ===


,Date,Team,Player,Opponent,Min,days_since_last_minutes,acute_workload_7d,match_count_14d,chronic_workload_28d,acwr
6072,2023-12-19,Newcastle United,Matt Ritchie,Chelsea,39,3,7.000,4,8.750,0.800
6073,2023-12-19,Newcastle United,Miguel Almirón,Chelsea,90,3,180.000,4,127.250,1.415
6074,2023-12-19,Newcastle United,Paul Dummett,Chelsea,0,24,0.000,4,1.250,0.000
6075,2023-12-19,Newcastle United,Sean Longstaff,Chelsea,90,3,110.000,4,34.250,3.212
6076,2023-12-19,Newcastle United,Sven Botman,Chelsea,45,3,7.000,4,1.750,4.000
6077,2023-12-19,Newcastle United,Valentino Livramento,Chelsea,90,3,173.000,4,129.000,1.341
6078,2023-12-23,Arsenal,Aaron Ramsdale,Liverpool,0,14,0.000,3,0.000,0.000
6079,2023-12-23,Arsenal,Ben White,Liverpool,90,6,90.000,3,89.250,1.008
6080,2023-12-23,Arsenal,Bukayo Saka,Liverpool,90,6,88.000,3,127.750,0.689
6081,2023-12-23,Arsenal,Cédric Soares,Liverpool,0,11,0.000,3,15.250,0.000


---
## 3. Load Data — SofaScore Position Profiles

In [54]:
all_seasonal_baselines = []

for season_str in seasons:
    all_players_path = BASE / 'Data' / season_str / 'sofascore' / 'premier_league' / f'premier_league_{season_str.replace("-", "_")}_all_players.csv'
    
    if not all_players_path.exists():
        print(f"[SKIP] Sofascore file not found for season: {season_str}")
        continue
        
    print(f"Processing Sofascore seasonal baselines: {season_str}...")
    df_raw = pd.read_csv(all_players_path)
    
    # 1. Filter out players with negligible game time to avoid skewed Per-90 anomalies
    df_filtered = df_raw[df_raw['minutesPlayed'] >= 90].copy()
    
    # Calculate the 90-minute multiplier factor
    df_filtered['90s_played'] = df_filtered['minutesPlayed'] / 90.0
    
    # --- COMBINED ML FEATURES ENGINE ---
    df_features = pd.DataFrame(index=df_filtered.index)
    df_features['player'] = df_filtered['player']
    df_features['team'] = df_filtered['team']
    df_features['season'] = season_str
    df_features['sofascore_rating_avg'] = df_filtered['rating']
    
    # A. ATTACK & CREATION PER 90
    df_features['xg_per90'] = df_filtered['expectedGoals'] / df_filtered['90s_played']
    df_features['xa_per90'] = df_filtered['expectedAssists'] / df_filtered['90s_played']
    df_features['key_passes_per90'] = df_filtered['keyPasses'] / df_filtered['90s_played']
    df_features['shots_per90'] = df_filtered['totalShots'] / df_filtered['90s_played']
    df_features['shot_accuracy'] = (df_filtered['shotsOnTarget'] / df_filtered['totalShots'].replace(0, np.nan)).fillna(0)
    
    # B. BALL RETENTION & PROGRESSION
    df_features['touches_per90'] = df_filtered['touches'] / df_filtered['90s_played']
    df_features['possession_loss_rate'] = (df_filtered['possessionLost'] / df_filtered['touches'].replace(0, np.nan)).fillna(0)
    df_features['dribble_success_rate'] = df_filtered['successfulDribblesPercentage'] / 100.0
    df_features['pass_accuracy'] = df_filtered['accuratePassesPercentage'] / 100.0
    
    # C. DEFENSIVE VOLUME & EFFICIENCY
    df_features['tackles_per90'] = df_filtered['tackles'] / df_filtered['90s_played']
    df_features['interceptions_per90'] = df_filtered['interceptions'] / df_filtered['90s_played']
    df_features['recoveries_per90'] = df_filtered['ballRecovery'] / df_filtered['90s_played']
    df_features['clearances_per90'] = df_filtered['clearances'] / df_filtered['90s_played']
    df_features['tackle_win_rate'] = df_filtered['tacklesWonPercentage'] / 100.0
    df_features['aerial_win_rate'] = df_filtered['aerialDuelsWonPercentage'] / 100.0
    
    # D. RISK MARKERS
    df_features['errors_to_shot_or_goal_per90'] = (df_filtered['errorLeadToShot'] + df_filtered['errorLeadToGoal']) / df_filtered['90s_played']
    df_features['fouls_conceded_per90'] = df_filtered['fouls'] / df_filtered['90s_played']
    
    # E. FIXED GOALKEEPER EXCLUSIVES
    # True Save Rate = Saves / (Saves + Goals Conceded)
    if 'saves' in df_filtered.columns and 'goalsConceded' in df_filtered.columns:
        total_shots_faced = df_filtered['saves'] + df_filtered['goalsConceded']
        df_features['gk_save_rate'] = (df_filtered['saves'] / total_shots_faced.replace(0, np.nan)).fillna(0)
    else:
        df_features['gk_save_rate'] = 0.0

    if 'goalsPrevented' in df_filtered.columns:
        df_features['gk_goals_prevented_per90'] = df_filtered['goalsPrevented'] / df_filtered['90s_played']
    else:
        df_features['gk_goals_prevented_per90'] = 0.0
        
    all_seasonal_baselines.append(df_features)

# Compile all seasons into a singular Master Historical Profiler
df_sofascore_player_profiles = pd.concat(all_seasonal_baselines, ignore_index=True)

print("\n=== SOFASCORE MASTER SEASONAL PROFILES COMPILED ===")
print(f"Total Profiles Created: {df_sofascore_player_profiles.shape[0]} rows")

Processing Sofascore seasonal baselines: 2022-2023...
Processing Sofascore seasonal baselines: 2023-2024...
Processing Sofascore seasonal baselines: 2024-2025...

=== SOFASCORE MASTER SEASONAL PROFILES COMPILED ===
Total Profiles Created: 1475 rows


In [62]:
# Check how many unique teams ended up in your final dataframe
for season_name, season_group in df_sofascore_player_profiles.groupby('season'):
    # Extract unique teams for this specific season and sort them alphabetically
    unique_teams_in_season = sorted(season_group['team'].unique())
    
    print(f"\nSeason: {season_name} (Total: {len(unique_teams_in_season)} teams)")
    print("-" * 50)
    
    # Print teams cleanly in bullet points
    print(f"  • {unique_teams_in_season}")


Season: 2022-2023 (Total: 20 teams)
--------------------------------------------------
  • ['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton & Hove Albion', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Leeds United', 'Leicester City', 'Liverpool FC', 'Manchester City', 'Manchester United', 'Newcastle United', 'Nottingham Forest', 'Southampton', 'Tottenham Hotspur', 'West Ham United', 'Wolverhampton']

Season: 2023-2024 (Total: 20 teams)
--------------------------------------------------
  • ['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton & Hove Albion', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Liverpool FC', 'Luton Town', 'Manchester City', 'Manchester United', 'Newcastle United', 'Nottingham Forest', 'Sheffield United', 'Tottenham Hotspur', 'West Ham United', 'Wolverhampton']

Season: 2024-2025 (Total: 20 teams)
--------------------------------------------------
  • ['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brig

In [ ]:

# This dictionary will hold our isolated position dataframes
# Structure will be: position_dfs['cm_mf'] -> Dataframe with all seasons
position_data_store = {}

for season_str in seasons:
    position_dir = BASE / 'Data' / season_str / 'sofascore' / 'premier_league' / 'position_profiles'
    
    if not position_dir.exists():
        continue
        
    for file_path in position_dir.glob('*.csv'):
        # Extract the position name (e.g., cm_mf, df_cb, gk)
        filename_parts = file_path.stem.split('_')
        pos_group = f"{filename_parts[-2]}_{filename_parts[-1]}" if len(filename_parts) >= 2 else filename_parts[-1]
        
        df_pos = pd.read_csv(file_path)
        df_pos.columns = df_pos.columns.str.strip()
        
        if 'team' not in df_pos.columns:
            continue
            
        df_pos['team'] = df_pos['team'].str.strip()
        df_pos['season'] = season_str
        
        # Initialize the list for this position group if it doesn't exist yet
        if pos_group not in position_data_store:
            position_data_store[pos_group] = []
            
        position_data_store[pos_group].append(df_pos)

# --- COMPILE INDIVIDUAL DATAFRAMES PER POSITION ---
final_position_dfs = {}

print("=== COMPILED POSITIONAL DATAFRAMES ===")
for pos_group, list_of_dfs in position_data_store.items():
    # Combine the seasons together for this specific position
    df_combined = pd.concat(list_of_dfs, ignore_index=True)
    
    # Reorder columns to put season and team up front
    cols = ['season', 'team'] + [c for c in df_combined.columns if c not in ['season', 'team']]
    df_combined = df_combined[cols]
    
    # Save it explicitly to our final dictionary
    final_position_dfs[pos_group] = df_combined
    print(f"- final_position_dfs['{pos_group}'] -> Shape: {df_combined.shape}")

=== COMPILED POSITIONAL DATAFRAMES ===
- final_position_dfs['cm_mf'] -> Shape: (1686, 15)
- final_position_dfs['df_cb'] -> Shape: (1686, 12)
- final_position_dfs['dm_cdm'] -> Shape: (1686, 13)
- final_position_dfs['fw_st'] -> Shape: (1686, 12)
- final_position_dfs['2023_gk'] -> Shape: (554, 7)
- final_position_dfs['wg_am'] -> Shape: (1686, 13)
- final_position_dfs['2024_gk'] -> Shape: (570, 7)
- final_position_dfs['2025_gk'] -> Shape: (562, 7)


In [71]:
# Pull out just the Center-Back profile table
df_center_backs = final_position_dfs['df_cb']

# View top 10 rows chronologically
df_center_backs.sort_values(by=['season', 'team']).head(10)

,season,team,appearances,minutesPlayed,tackles,interceptions,clearances,aerialDuelsWon,groundDuelsWon,blockedShots,errorLeadToGoal,rating
18,2022-2023,Arsenal,38,3426,1,0,34,7,5,0,2,6.860
58,2022-2023,Arsenal,38,3417,48,32,120,100,90,6,1,7.010
67,2022-2023,Arsenal,27,2416,34,19,82,53,56,0,1,6.960
75,2022-2023,Arsenal,27,2133,40,22,31,51,66,8,0,6.930
83,2022-2023,Arsenal,38,3079,60,22,75,33,98,2,0,6.910
142,2022-2023,Arsenal,7,431,7,6,10,8,9,0,0,6.740
158,2022-2023,Arsenal,14,587,8,3,26,22,11,2,0,6.680
159,2022-2023,Arsenal,21,680,27,14,14,17,31,2,1,6.680
172,2022-2023,Arsenal,27,793,20,11,20,6,31,1,0,6.650
222,2022-2023,Arsenal,37,3144,36,6,12,14,110,33,1,7.350


In [72]:
all_injury_records = []

for season_str in seasons:
    injury_path = BASE / 'Data' / season_str / 'injuries' / f'ALL_TEAMS_{season_str}_injuries_days_out.csv'
    
    if not injury_path.exists():
        print(f"[SKIP] Injury file not found for season: {season_str}")
        continue
        
    print(f"Processing Injury logs for season: {season_str}...")
    df_inj = pd.read_csv(injury_path)
    
    # Clean up column spaces
    df_inj.columns = df_inj.columns.str.strip()
    
    # Standardize column naming conventions to lowercase keys
    # (Adjust 'player' or 'team' mapping here if your CSV columns use different names like 'Player Name')
    rename_dict = {}
    for col in df_inj.columns:
        if col.lower() in ['player', 'player_name', 'name']:
            rename_dict[col] = 'player'
        elif col.lower() in ['team', 'club']:
            rename_dict[col] = 'team'
            
    df_inj = df_inj.rename(columns=rename_dict)
    df_inj['season'] = season_str
    
    # Ensure any strings are clean of trailing spaces
    if 'player' in df_inj.columns:
        df_inj['player'] = df_inj['player'].str.strip()
    if 'team' in df_inj.columns:
        df_inj['team'] = df_inj['team'].str.strip()
        
    all_injury_records.append(df_inj)

if len(all_injury_records) > 0:
    df_master_injuries = pd.concat(all_injury_records, ignore_index=True)
    print("\n=== MASTER INJURY LOGS INSTANTIATED ===")
    print(f"Total Recorded Injury Instances: {df_master_injuries.shape[0]} rows")
    print("\nColumns found in your injury data:")
    print(list(df_master_injuries.columns))
else:
    print("[ERROR] No injury data could be processed.")

Processing Injury logs for season: 2022-2023...
Processing Injury logs for season: 2023-2024...
Processing Injury logs for season: 2024-2025...

=== MASTER INJURY LOGS INSTANTIATED ===
Total Recorded Injury Instances: 9526 rows

Columns found in your injury data:
['season', 'team_name', 'player_id', 'player', 'injury_type', 'reason', 'fixture_date', 'end_date', 'days_out']


In [78]:
# Ensure fixture_date is a datetime object
df_master_injuries['fixture_date'] = pd.to_datetime(df_master_injuries['fixture_date'])

# Normalize names to match your core dataframe keys
df_master_injuries = df_master_injuries.rename(columns={
    'player_name': 'player',
    'team_name': 'team'
})

df_master_injuries['player'] = df_master_injuries['player'].str.strip()
df_master_injuries['team'] = df_master_injuries['team'].str.strip()

# --- STEP 2: EXPAND ROWS WITH FALLBACK LOGIC ---
injury_days_list = []

for idx, row in df_master_injuries.dropna(subset=['fixture_date', 'days_out']).iterrows():
    start_date = row['fixture_date']
    days_out_count = int(row['days_out'])
    
    # If days_out is 0 (e.g., a tiny knock), we count it as at least 1 day unavailable
    if days_out_count <= 0:
        days_out_count = 1
        
    # Programmatically calculate end_date as a fallback
    calculated_end_date = start_date + pd.Timedelta(days=days_out_count - 1)
    
    # Generate the full date range sequence
    date_range = pd.date_range(start=start_date, end=calculated_end_date)
    total_days = len(date_range)
    
    for i, single_date in enumerate(date_range):
        days_remaining = total_days - i - 1
        injury_days_list.append({
            'season': row['season'],
            'team': row['team'],
            'player': row['player'],
            'Date': single_date,
            'is_injured': 1,
            'days_remaining_out': days_remaining,
            'injury_type': row['injury_type'],
            'injury_reason': row['reason']
        })

# Create the daily medical lookup table safely
if len(injury_days_list) > 0:
    df_daily_injury_lookup = pd.DataFrame(injury_days_list)

    # Clean up duplicate dates per player cleanly
    df_daily_injury_lookup = df_daily_injury_lookup.sort_values('days_remaining_out', ascending=False)
    df_daily_injury_lookup = df_daily_injury_lookup.drop_duplicates(subset=['season', 'team', 'player', 'Date'], keep='first')

    print("=== DAILY INJURY TIMELINE LOOKUP GENERATED ===")
    print(f"Total Active Injury Days Tracked: {df_daily_injury_lookup.shape[0]} rows")
else:
    print("[ERROR] No rows generated. Please double check your 'fixture_date' and 'days_out' columns for valid data.")

=== DAILY INJURY TIMELINE LOOKUP GENERATED ===
Total Active Injury Days Tracked: 206646 rows


In [79]:
df_daily_injury_lookup.head(10)

,season,team,player,Date,is_injured,days_remaining_out,injury_type,injury_reason
274380,2022-2023,Manchester City,B. Mendy,2023-02-12,1,306,Missing Fixture,Suspended
282034,2022-2023,Manchester City,B. Mendy,2023-05-17,1,306,Missing Fixture,Suspended
270036,2022-2023,Manchester City,B. Mendy,2022-10-29,1,306,Missing Fixture,Suspended
262203,2022-2023,Manchester City,B. Mendy,2022-08-07,1,306,Missing Fixture,Suspended
281227,2022-2023,Manchester City,B. Mendy,2023-05-09,1,306,Missing Fixture,Suspended
279240,2022-2023,Manchester City,B. Mendy,2023-04-19,1,306,Missing Fixture,Suspended
273288,2022-2023,Manchester City,B. Mendy,2023-01-19,1,306,Missing Fixture,Suspended
262709,2022-2023,Manchester City,B. Mendy,2022-08-13,1,306,Missing Fixture,Suspended
273595,2022-2023,Manchester City,B. Mendy,2023-01-22,1,306,Missing Fixture,Suspended
269628,2022-2023,Manchester City,B. Mendy,2022-10-25,1,306,Missing Fixture,Suspended


In [82]:
print("=== INJURY DATA SANITY CHECK ===")

# 1. Check for negative or suspicious days out
negative_days = df_master_injuries[df_master_injuries['days_out'] <= 0]
print(f"• Entries with 0 or negative days_out: {len(negative_days)}")

# 2. Check for extreme outliers (e.g., injuries lasting more than 6 months)
extreme_days = df_master_injuries[df_master_injuries['days_out'] > 180]
print(f"• Entries with > 180 days_out: {len(extreme_days)}")

# 3. Check for missing critical values
missing_dates = df_master_injuries['fixture_date'].isna().sum()
print(f"• Entries missing a fixture_date: {missing_dates}")


=== INJURY DATA SANITY CHECK ===
• Entries with 0 or negative days_out: 471
• Entries with > 180 days_out: 4147
• Entries missing a fixture_date: 0


---
## 4. Descriptive Statistics & Coefficient of Variation

### 📝 Section 4 Conclusions — Descriptive Statistics

*Write your observations here after running the cell. Template:*

- **Rest days** — mean ≈ X days, but high CV (>60%) confirms large variability in fixture spacing; congested weeks are concentrated in specific calendar periods.
- **Goals for/against** — low mean but high skew suggests most matches are low-scoring with rare high-scoring outliers.
- **Pass accuracy** (SofaScore) — relatively low CV, meaning this is a stable player trait rather than match-to-match noise.
- **Dribble success** — high CV indicates this varies enormously by player type and position, not just form.
- **Key passes & recoveries** — high skewness: a small number of players contribute disproportionately.

---
## 5. Missing Value Analysis

---
## 6. Outlier Detection — IQR Method

### 📝 Section 6 Conclusions — Outlier Detection

- **Rest days** — extreme outliers (>21 days) correspond to international breaks or player injuries; these are real events, not data errors. Flagged as `long_gap_return` in the feature set.
- **Goals per match** — upper outliers (≥4 goals) are rare but genuine events; no removal needed.
- **Player minutes** — some players appear with 0 or very low minutes (substitutes, cup appearances); filtered by the ≥180 min threshold.
- **SofaScore rating** — values near 10 (maximum) or below 5 are genuine outlier performances, not measurement errors.

---
## 7. Histogram Visualisation — Key Variables

---
## 8. Correlation Matrix & Heatmap

### 📝 Section 8 Conclusions — Correlations

- **xG ↔ shots**: expected strong positive correlation — xG is a function of shot quality and quantity.
- **Rest days ↔ points / goal_diff**: if negative (more rest → worse results), this would be counter-intuitive and worth investigating; more likely we see that short rest shows a small negative effect on goal_diff.
- **matches_last_7d ↔ points**: the key congestion effect variable — a negative correlation here supports the research hypothesis.
- **key_passes ↔ assists**: strong correlation (key passes are the precursor to assists).
- **tackles ↔ interceptions**: high correlation — defensive players tend to score high on both.

---
## 9. Radar Charts — Position Performance Profiles

### 📝 Section 9 Conclusions — Radar Charts

- Teams with strong UCL campaigns (Arsenal, Liverpool) typically show **balanced** attacking and defensive profiles — depth is required across all metrics.
- Manchester City's radar should show a distinctive possession-oriented shape (high pass accuracy, lower defensive duels compared to pressing teams like Liverpool).
- Season-on-season radars for a single team reveal whether UCL participation depletes specific qualities (e.g. dribble success dropping in UCL-heavy seasons).

---
## 10. Lorenz Curve & Gini Coefficient — Workload Inequality

### 📝 Section 10 Conclusions — Lorenz / Gini

- A **high Gini** for minutes means the manager relies heavily on a small group of players → higher individual fatigue risk.
- UCL teams are expected to show **lower Gini** than the control group, since rotation is a necessity — but this may not hold if the manager has a fixed XI.
- xG Gini reveals **attacking dependency**: teams with one dominant forward will show high xG concentration — this is a risk factor if that player is fatigued or injured.
- Comparing Gini across seasons reveals whether UCL participation forces managers to rotate more.

---
## 11. Compute New Attributes — Fatigue & Performance Indicators

---
## 12. Combine All Sources — Master Analysis Dataset

---
## 13. Cross-Source Relationship Analysis

### 📝 Section 13 Conclusions — Cross-Source Relationships

- **Short-rest % vs. points**: the regression slope direction tells you whether congestion hurts results. A negative slope confirms the hypothesis; a flat/positive slope would challenge it (depth of squad matters).
- **FBref composite vs. SofaScore rating**: if r > 0.6, the two sources are measuring the same underlying reality — good for model confidence. Positions with low r (e.g. GK) may need position-specific scoring.
- **Congestion vs. xG per 90**: if players on congested teams show lower per-90 xG despite similar raw stats, it suggests fatigue reduces shot quality / positioning rather than volume.

---
## 14. Final EDA Summary & Key Findings

---
## 📝 Final Written Conclusions

### 1. Descriptive Statistics
Across 3 seasons, UCL teams average significantly more matches per season (55–60) than non-UCL PL peers (~38). Rest days distribution is right-skewed: the median is around 7 days, but the tail extends to 30+ (international breaks), while the dangerous short-rest window (≤4 days) represents roughly 20–30% of all matches for UCL teams.

### 2. Missing Values
Primary sources of missingness are: (i) rest_days for a team's first match of the season — structurally absent, handled as NaN; (ii) some FBref advanced stats for cup matches; (iii) SofaScore ratings for players with fewer than 3 appearances. No column exceeds 15% missingness in the core numeric features used for modelling.

### 3. Outliers
Rest-day outliers (>21 days) correspond to international breaks or injuries — real events kept in the dataset but flagged with `long_gap_return`. Performance score outliers (extreme games) are genuine and retained; IQR fences are used as soft guidance, not hard filters.

### 4. Distributions
Goals, xG, and defensive metrics are right-skewed (most matches/players near zero, few extreme events). Pass accuracy is approximately normal. Congestion score shows bimodal distribution: normal weeks vs. congested periods around UCL knockout rounds.

### 5. Correlations
The strongest correlations: xG↔shots_on_target (r≈0.85), key_passes↔assists (r≈0.7), tackles↔interceptions (r≈0.65). The congestion×performance relationship shows a small but consistent negative correlation between `matches_last_7d` and `goal_diff` (r≈−0.12 to −0.18), supporting the research hypothesis but suggesting other confounders.

### 6. Radar Charts
Team profiles confirm expected stylistic differences: City is possession-dominant (high pass_pct, low aerial duel count); Liverpool is press-intensive (high pressures, high recoveries); Arsenal is balanced. UCL participation does not uniformly depress team profiles — squad depth appears to maintain averages.

### 7. Lorenz / Gini
Playing time Gini coefficients range from ~0.25 (well-rotated squads) to ~0.45 (heavily dependent on a core XI). Teams with higher Gini are more vulnerable to player fatigue — concentration of minutes in key players amplifies the risk when those players are congested. This is a novel variable worth including in the XGBoost model.

### 8. New Attributes
The ACWR is the most promising new fatigue proxy — it captures the spike in load relative to recent baseline, which is the mechanism behind sports science injury risk models. `congestion_score` and `travel_stress` add contextual nuance. The composite performance score (FBref) correlates well with SofaScore rating (validation check), confirming it as a valid target variable.